# M5 Full Backtest Report
所有 6 个 pattern 综合回测 — 分组分析 + 置信度校准 + 学习曲线

In [1]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from spx_scanner.data_layer.loader import load_data
from spx_scanner.data_layer.resampler import resample_1m_to_3m
from spx_scanner.features import compute_all_features
from spx_scanner.scanner.engine import ScannerEngine
from spx_scanner.patterns.registry import get_all_patterns
from spx_scanner.backtest.simulator import run_backtest
from spx_scanner.backtest.metrics import (
    compute_metrics, confidence_calibration, learning_curve
)
from spx_scanner.backtest.grouping import (
    enrich_trades, group_by_segment, group_by_vix,
    group_by_dow, group_by_confidence, session_heatmap,
    confidence_win_rate_table,
)

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ── 加载数据 & 特征 ────────────────────────────────────────────────────────
df1m = load_data('../data/spy_1min.parquet')
df3m = resample_1m_to_3m(df1m)
df   = compute_all_features(df3m)
print(f'3min bars: {len(df)}  columns: {len(df.columns)}')

3min bars: 1820  columns: 60


In [3]:
# ── 扫描所有 pattern ───────────────────────────────────────────────────────
patterns = get_all_patterns('SPY')
engine   = ScannerEngine(patterns=patterns)
sigs_df  = engine.scan(df)
print(f'Total signals: {len(sigs_df)}')
print(sigs_df.groupby(['pattern','direction']).size().to_string())

Total signals: 290
pattern          direction
failed_breakout  call          1
                 put          19
last_hour_drift  call         81
                 put          31
liquidity_sweep  call          5
                 put          11
orb_breakout     call         75
                 put          56
squeeze_release  call          5
vwap_rejection   call          1
                 put           5


In [4]:
# ── 回测(3 种退出策略) ────────────────────────────────────────────────────
trades = run_backtest(df, sigs_df)
print(f'Total trades: {len(trades)}')
print(trades.groupby(['pattern','exit_strategy']).size().unstack(fill_value=0).to_string())

Total trades: 870
exit_strategy    fixed_time  target_stop  trailing_atr
pattern                                               
failed_breakout          20           20            20
last_hour_drift         112          112           112
liquidity_sweep          16           16            16
orb_breakout            131          131           131
squeeze_release           5            5             5
vwap_rejection            6            6             6


In [5]:
# ── 基础指标表 ─────────────────────────────────────────────────────────────
metrics = compute_metrics(trades)
display_cols = ['n_trades','win_rate','expect_pct','profit_factor','sharpe',
                'max_drawdown_pct','max_consec_wins','max_consec_losses','calmar']
print(metrics[display_cols].to_string())

                               n_trades  win_rate  expect_pct  profit_factor  sharpe  max_drawdown_pct  max_consec_wins  max_consec_losses   calmar
pattern         exit_strategy                                                                                                                      
failed_breakout fixed_time           20     0.450     -0.0183          0.724  -0.389           -0.8582                8                  5  -11.525
                target_stop          20     0.450     -0.0178          0.729  -0.383           -0.8490                8                  5  -11.356
                trailing_atr         20     0.250     -0.0538          0.258  -1.261           -0.9504                2                  6  -30.659
last_hour_drift fixed_time          112     0.598      0.0012          1.028   0.111           -2.2580               21                 12    0.052
                target_stop         112     0.562     -0.0003          0.994  -0.026           -2.6023          

In [6]:
# ── 环境富化 ───────────────────────────────────────────────────────────────
enriched = enrich_trades(trades, df)
print(f'Context columns added: {[c for c in enriched.columns if c not in trades.columns]}')
print(enriched[['pattern','session_segment','vix_regime','dow','confidence_bucket']].head(8).to_string())

Context columns added: ['session_segment', 'vix_regime', 'dow', 'is_opex', 'is_fomc', 'is_cpi', 'confidence_bucket']
           pattern session_segment vix_regime     dow confidence_bucket
0  last_hour_drift           close    unknown  Monday           0.6-0.7
1  last_hour_drift           close    unknown  Monday           0.6-0.7
2  last_hour_drift           close    unknown  Monday           0.6-0.7
3  last_hour_drift           close    unknown  Monday           0.6-0.7
4  last_hour_drift           close    unknown  Monday           0.6-0.7
5  last_hour_drift           close    unknown  Monday           0.6-0.7
6  last_hour_drift           close    unknown  Monday           0.6-0.7
7  last_hour_drift           close    unknown  Monday           0.6-0.7


In [7]:
# ── 分组分析 ───────────────────────────────────────────────────────────────
# 只看 target_stop 策略(更实际)
ts_trades = enriched[enriched['exit_strategy'] == 'target_stop']

seg_tbl  = group_by_segment(ts_trades, by_pattern=True)
vix_tbl  = group_by_vix(ts_trades, by_pattern=True)
dow_tbl  = group_by_dow(ts_trades)
conf_tbl = group_by_confidence(ts_trades, by_pattern=True)

print('=== Session Segment ===')
print(seg_tbl[['n_trades','win_rate','expect_pct']].to_string())
print('\n=== VIX Regime ===')
print(vix_tbl[['n_trades','win_rate','expect_pct']].to_string())
print('\n=== Day of Week ===')
print(dow_tbl[['n_trades','win_rate','expect_pct']].to_string())

=== Session Segment ===
                                 n_trades  win_rate  expect_pct
session_segment pattern                                        
close           failed_breakout         7     0.571     -0.0750
                last_hour_drift       112     0.562     -0.0003
                liquidity_sweep         2     0.000     -0.1322
                orb_breakout           29     0.448      0.0377
                squeeze_release         2     0.500      0.0049
                vwap_rejection          1     0.000     -0.1200
midday          failed_breakout        13     0.385      0.0130
                liquidity_sweep        13     0.462      0.0124
                orb_breakout           93     0.441     -0.0307
                squeeze_release         3     0.333     -0.0437
                vwap_rejection          3     0.667      0.0130
open            liquidity_sweep         1     1.000      0.0508
                orb_breakout            9     0.778     -0.1069
                

In [8]:
# ── 置信度校准 ─────────────────────────────────────────────────────────────
calib = confidence_calibration(ts_trades, n_bins=5)
print('=== Confidence Calibration ===')
print(calib.to_string())

=== Confidence Calibration ===
   bin_low  bin_high  n_trades  actual_win_rate  sufficient
0      0.5       0.6        12            0.083        True
1      0.6       0.7        65            0.523        True
2      0.7       0.8       208            0.524        True
3      0.8       0.9         4            0.250        True
4      0.9       1.0         1            1.000       False


In [9]:
# ── 学习曲线 ───────────────────────────────────────────────────────────────
lc = learning_curve(ts_trades, n_blocks=5)
print('=== Learning Curve ===')
print(lc.to_string())

=== Learning Curve ===
  block_start   block_end  n_trades  win_rate  total_pnl_pct
0  2026-04-06  2026-04-08        58     0.483         0.0307
1  2026-04-08  2026-04-13        58     0.655         2.5010
2  2026-04-14  2026-04-15        58     0.586         0.8065
3  2026-04-15  2026-04-20        58     0.431        -0.8869
4  2026-04-21  2026-04-23        58     0.362        -5.2955


In [10]:
# ── 综合图表 ───────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('M5 Full Backtest Report — SPY 3min (14 days)', fontsize=14, fontweight='bold')

# 1. 胜率 by (pattern, exit_strategy)
ax1 = fig.add_subplot(gs[0, 0])
wr_pivot = metrics['win_rate'].unstack('exit_strategy')
wr_pivot.plot(kind='bar', ax=ax1, edgecolor='white')
ax1.set_title('Win Rate by Pattern & Exit Strategy')
ax1.set_ylim(0, 1)
ax1.axhline(0.5, color='red', ls='--', lw=0.8)
ax1.tick_params(axis='x', rotation=30)
ax1.legend(fontsize=7)

# 2. 期望值 by (pattern, exit_strategy)
ax2 = fig.add_subplot(gs[0, 1])
exp_pivot = metrics['expect_pct'].unstack('exit_strategy')
exp_pivot.plot(kind='bar', ax=ax2, edgecolor='white')
ax2.set_title('Expected P&L % by Pattern & Exit')
ax2.axhline(0, color='red', ls='--', lw=0.8)
ax2.tick_params(axis='x', rotation=30)
ax2.legend(fontsize=7)

# 3. 最大回撤
ax3 = fig.add_subplot(gs[0, 2])
dd_pivot = metrics['max_drawdown_pct'].unstack('exit_strategy')
dd_pivot.plot(kind='bar', ax=ax3, edgecolor='white')
ax3.set_title('Max Drawdown % by Pattern & Exit')
ax3.tick_params(axis='x', rotation=30)
ax3.legend(fontsize=7)

# 4. Session 热力图(胜率)
ax4 = fig.add_subplot(gs[1, 0])
hm = session_heatmap(ts_trades, value='win_rate')
if not hm.empty:
    im = ax4.imshow(hm.values.astype(float), cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    ax4.set_xticks(range(len(hm.columns)))
    ax4.set_xticklabels(hm.columns, fontsize=8)
    ax4.set_yticks(range(len(hm.index)))
    ax4.set_yticklabels(hm.index, fontsize=7)
    for i in range(len(hm.index)):
        for j in range(len(hm.columns)):
            v = hm.values[i, j]
            if not np.isnan(v):
                ax4.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7)
    plt.colorbar(im, ax=ax4)
ax4.set_title('Win Rate: Pattern × Session')

# 5. VIX regime 胜率
ax5 = fig.add_subplot(gs[1, 1])
if not vix_tbl.empty and 'win_rate' in vix_tbl.columns:
    vix_wr = vix_tbl['win_rate'].unstack('pattern') if vix_tbl.index.nlevels == 2 else vix_tbl['win_rate']
    vix_wr.plot(kind='bar', ax=ax5, edgecolor='white')
    ax5.axhline(0.5, color='red', ls='--', lw=0.8)
    ax5.set_ylim(0, 1)
ax5.set_title('Win Rate by VIX Regime')
ax5.tick_params(axis='x', rotation=0)
ax5.legend(fontsize=7)

# 6. Day-of-week 胜率
ax6 = fig.add_subplot(gs[1, 2])
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday']
if not dow_tbl.empty and 'win_rate' in dow_tbl.columns:
    dow_wr = dow_tbl['win_rate'].reindex([d for d in dow_order if d in dow_tbl.index])
    ax6.bar(range(len(dow_wr)), dow_wr.values, color='steelblue', edgecolor='white')
    ax6.set_xticks(range(len(dow_wr)))
    ax6.set_xticklabels([d[:3] for d in dow_wr.index], fontsize=9)
    ax6.axhline(0.5, color='red', ls='--', lw=0.8)
    ax6.set_ylim(0, 1)
ax6.set_title('Win Rate by Day of Week')

# 7. 置信度校准曲线
ax7 = fig.add_subplot(gs[2, 0])
if not calib.empty:
    suf = calib[calib['sufficient']]
    insuf = calib[~calib['sufficient']]
    mid = (calib['bin_low'] + calib['bin_high']) / 2
    suf_mid = (suf['bin_low'] + suf['bin_high']) / 2
    insuf_mid = (insuf['bin_low'] + insuf['bin_high']) / 2
    ax7.plot([calib['bin_low'].min(), calib['bin_high'].max()],
             [calib['bin_low'].min(), calib['bin_high'].max()],
             'k--', lw=0.8, label='Perfect calibration')
    ax7.scatter(suf_mid, suf['actual_win_rate'], s=60, zorder=5, label='Sufficient samples')
    ax7.scatter(insuf_mid, insuf['actual_win_rate'], s=40, marker='x', color='gray', label='Insufficient')
    ax7.set_xlabel('Model Confidence')
    ax7.set_ylabel('Actual Win Rate')
    ax7.legend(fontsize=7)
ax7.set_title('Confidence Calibration (target_stop)')

# 8. 学习曲线
ax8 = fig.add_subplot(gs[2, 1])
if not lc.empty:
    ax8.bar(range(len(lc)), lc['win_rate'], color='steelblue', edgecolor='white')
    ax8.set_xticks(range(len(lc)))
    ax8.set_xticklabels([str(b)[:10] for b in lc['block_start']], rotation=20, fontsize=7)
    ax8.axhline(0.5, color='red', ls='--', lw=0.8)
    ax8.set_ylim(0, 1)
    ax8.set_ylabel('Win Rate')
ax8.set_title('Learning Curve (Win Rate by Time Block)')

# 9. 累积 P&L 曲线
ax9 = fig.add_subplot(gs[2, 2])
for pat, grp in ts_trades.groupby('pattern'):
    cum_pnl = grp.sort_values('entry_time')['pnl_pct'].cumsum()
    ax9.plot(range(len(cum_pnl)), cum_pnl.values, label=pat, lw=1.2)
ax9.axhline(0, color='black', ls='--', lw=0.5)
ax9.set_xlabel('Trade #')
ax9.set_ylabel('Cumulative P&L %')
ax9.legend(fontsize=7)
ax9.set_title('Cumulative P&L by Pattern (target_stop)')

plt.savefig('M5_full_backtest.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved M5_full_backtest.png')

Saved M5_full_backtest.png


In [11]:
# ── OOS 警告:数据量统计 ─────────────────────────────────────────────────────
from spx_scanner.config_loader import load_config
cfg = load_config()
min_samples = cfg['backtest']['min_samples_per_pattern']

print('=== Pattern Sample Count vs Min Required ===')
cnts = ts_trades.groupby('pattern').size()
for pat, cnt in cnts.items():
    flag = '✓' if cnt >= min_samples else '⚠ INSUFFICIENT'
    print(f'  {pat:25s}: {cnt:4d} trades  {flag}')

print(f'\nMin required per pattern: {min_samples}')
print('Note: 14 days of data is too short for full OOS validation → covered in M7.')

=== Pattern Sample Count vs Min Required ===
  failed_breakout          :   20 trades  ⚠ INSUFFICIENT
  last_hour_drift          :  112 trades  ✓
  liquidity_sweep          :   16 trades  ⚠ INSUFFICIENT
  orb_breakout             :  131 trades  ✓
  squeeze_release          :    5 trades  ⚠ INSUFFICIENT
  vwap_rejection           :    6 trades  ⚠ INSUFFICIENT

Min required per pattern: 30
Note: 14 days of data is too short for full OOS validation → covered in M7.


## M5 Summary

### Deliverables
- `backtest/metrics.py` — 扩展全套指标: max_drawdown, calmar, consec wins/losses, confidence_calibration, learning_curve
- `backtest/grouping.py` — 分组函数: segment / vix / dow / confidence / heatmap
- `tests/test_backtest_m5.py` — 41 tests
- **186 tests total, 0 failures**

### Key Observations
- Data: 14 days, 290 raw signals → ~290 trades per strategy × 3 strategies
- Most patterns have **insufficient samples** for statistical conclusions (< 30 per pattern in test set)
- Full OOS validation requires more data → M7

→ M6 next: Streamlit dashboard